### 3.2.5 Search: 检索API
① 源码剖析

函数签名如下

```python
search(
  self,
  namespace_prefix: tuple[str, ...],
  /,
  *,
  query: str | None = None,
  filter: dict[str, Any] | None = None,
  limit: int = 10,
  offset: int = 0,
  refresh_ttl: bool | None = None
) -> list[SearchItem]
```

它支持两种检索方式（对应上面的参数2、3）：
1. 按 filter 做结构化过滤 ，即用 value 中的 键值 筛选符合条件的记录。
2. 按 query 做语义相似度检索 ，需要将输入转换为向量。

返回值：
* 返回匹配的 SearchItem 列表，并额外携带匹配分数等检索元信息。

参数说明:

| 参数 | 类型 | 含义 |
|---|---|---|
| `namespace_prefix` | `tuple[str, ...]` | namespace 的**前缀**，按“段”比较：`("users",)` 匹配所有用户，`("users", "Bob")` 只匹配 Bob，`("users", "Bo")` 匹配不到 Bob。`()` 匹配全部 |
| `query` | `str` / `None` | 自然语言查询，做语义检索。只有创建 store 时配置了 `index` 才生效；没配置时会被**直接忽略**，不报错，`score` 全为 `None` |
| `filter` | `dict` / `None` | 按 value 里的字段过滤，规则见“举例2：按照filter过滤”后面的说明 |
| `limit` | `int` | 最多返回几条，默认 10 |
| `offset` | `int` | 跳过前几条，和 `limit` 配合分页 |
| `refresh_ttl` | `bool` / `None` | 同 `get()`：只对设置了 TTL 的条目，读取时是否让过期时间重新计时 |

签名里 `/` 前面的参数只能按位置传（不能写成 `namespace_prefix=...`），`*` 后面的参数只能按关键字传。

返回值 `list[SearchItem]`：

* `SearchItem` 是 `Item` 的子类，多一个 `score` 字段。`print` 出来显示的是 `Item(...)`，因为 `__repr__` 继承自 Item，里面写死了 `"Item("`，实际类型是 SearchItem。
* `score`：带 query 时是 query 和记忆的**余弦相似度**，结果按 score 从高到低排列；不带 query 时为 `None`。
* 不带 query 时的顺序：InMemoryStore 按写入顺序；SqliteStore / PostgresStore 按 `updated_at` 倒序。
* 没有匹配时返回空列表 `[]`。

#### 3.2.5.1 以InMemoryStore 测试

In [1]:
from langgraph.store.memory import InMemoryStore

store = InMemoryStore()

namespace1 = ("users", "Alice", "memories")
key1 = 'preferences'
value1 = {
    "course": "计算机组成原理",
    "sports": "跑步",
    "food": "紫光园奶皮子酸奶"
}

namespace2 = ("users", "Bob", "memories")
key2 = 'preferences'
value2 = {
    "course": "数字电路与模拟电路",
    "sports": "跑步",
    "food": "奶皮子糖葫芦"
}

namespace3 = ("users", "Black", "memories")
key3 = 'preferences'
value3 = {
    "course": "数字电路与模拟电路",
    "sports": "羽毛球",
    "food": "紫光园奶皮子酸奶"
}

store.put(namespace1, key1, value1)
store.put(namespace2, key2, value2)
store.put(namespace3, key3, value3)

for item in store.search(("users",)):
    print(item)

Item(namespace=['users', 'Alice', 'memories'], key='preferences', value={'course': '计算机组成原理', 'sports': '跑步', 'food': '紫光园奶皮子酸奶'}, created_at='2026-10-08T12:53:07.592947+00:00', updated_at='2026-10-08T12:53:07.592950+00:00', score=None)
Item(namespace=['users', 'Bob', 'memories'], key='preferences', value={'course': '数字电路与模拟电路', 'sports': '跑步', 'food': '奶皮子糖葫芦'}, created_at='2026-10-08T12:53:07.592962+00:00', updated_at='2026-10-08T12:53:07.592963+00:00', score=None)
Item(namespace=['users', 'Black', 'memories'], key='preferences', value={'course': '数字电路与模拟电路', 'sports': '羽毛球', 'food': '紫光园奶皮子酸奶'}, created_at='2026-10-08T12:53:07.592971+00:00', updated_at='2026-10-08T12:53:07.592972+00:00', score=None)


In [2]:
for item in store.search(("users","Bob")):
    print(item)

Item(namespace=['users', 'Bob', 'memories'], key='preferences', value={'course': '数字电路与模拟电路', 'sports': '跑步', 'food': '奶皮子糖葫芦'}, created_at='2026-10-08T12:53:07.592962+00:00', updated_at='2026-10-08T12:53:07.592963+00:00', score=None)


举例2：按照filter过滤

In [3]:
for item in store.search(("users",),filter={"food": "紫光园奶皮子酸奶"}):
    print(item)

Item(namespace=['users', 'Alice', 'memories'], key='preferences', value={'course': '计算机组成原理', 'sports': '跑步', 'food': '紫光园奶皮子酸奶'}, created_at='2026-10-08T12:53:07.592947+00:00', updated_at='2026-10-08T12:53:07.592950+00:00', score=None)
Item(namespace=['users', 'Black', 'memories'], key='preferences', value={'course': '数字电路与模拟电路', 'sports': '羽毛球', 'food': '紫光园奶皮子酸奶'}, created_at='2026-10-08T12:53:07.592971+00:00', updated_at='2026-10-08T12:53:07.592972+00:00', score=None)


In [4]:
for item in store.search(("users",),filter={"sports": "跑步"}):
    print(item)

Item(namespace=['users', 'Alice', 'memories'], key='preferences', value={'course': '计算机组成原理', 'sports': '跑步', 'food': '紫光园奶皮子酸奶'}, created_at='2026-10-08T12:53:07.592947+00:00', updated_at='2026-10-08T12:53:07.592950+00:00', score=None)
Item(namespace=['users', 'Bob', 'memories'], key='preferences', value={'course': '数字电路与模拟电路', 'sports': '跑步', 'food': '奶皮子糖葫芦'}, created_at='2026-10-08T12:53:07.592962+00:00', updated_at='2026-10-08T12:53:07.592963+00:00', score=None)


**filter 的规则**

filter 只能针对 **value 字典里的字段**：InMemoryStore 的实现是取 `item.value.get(字段名)`，再和给定的值比较。namespace 只能用 `namespace_prefix` 限定，key、`created_at`、`updated_at` 都不能用 filter 过滤。

除了精确相等，还有几种写法（源码 `_compare_values`）：

| 写法 | 含义 |
|---|---|
| `{"sports": "跑步"}` | 等于，字符串、数字、布尔值都可以 |
| `{"age": {"$gt": 25}}` | 比较运算：`$eq` `$ne` `$gt` `$gte` `$lt` `$lte`；同一个字段可以写多个，如 `{"$gte": 20, "$lt": 30}` |
| `{"profile": {"city": "北京"}}` | value 里的嵌套字典：只比较写出来的子字段 |
| `{"tags": ["x", "y"]}` | 列表必须整个相等（长度、顺序都一样），不是“包含” |
| 写多个字段 | 必须全部满足（AND） |

不支持：OR、`$in`（会报 `ValueError: Unsupported operator`）、子串匹配、“列表包含某个元素”。

两个容易踩的坑：

* 字段不存在时取到的是 `None`，所以 `{"不存在的字段": None}` 会匹配所有条目。
* 嵌套字段的写法各 store 不同：InMemoryStore 要写成嵌套字典 `{"profile": {"city": "北京"}}`，写成 `{"profile.city": "北京"}` 匹配不到；SqliteStore 正好相反，点号路径可以，嵌套字典会报错。

In [ ]:
# 不等于：sports 不是"跑步"的记忆
for item in store.search(("users",), filter={"sports": {"$ne": "跑步"}}):
    print(item.namespace, item.value)

print("-" * 20)

# 多个字段：两个条件同时满足
for item in store.search(("users",), filter={"sports": "跑步", "food": "紫光园奶皮子酸奶"}):
    print(item.namespace, item.value)

('users', 'Black', 'memories') {'course': '数字电路与模拟电路', 'sports': '羽毛球', 'food': '紫光园奶皮子酸奶'}
--------------------
('users', 'Alice', 'memories') {'course': '计算机组成原理', 'sports': '跑步', 'food': '紫光园奶皮子酸奶'}


举例3: 按照语义搜索

**语义搜索的原理**

filter 要求字面相等，语义搜索找的是“意思相近”。做法是：

* **嵌入模型**把一段文本变成一个**向量**，即一串长度固定的数字（举例2 的模型是 1024 个）。模型用大量文本训练过，意思相近的文本会得到方向相近的向量。
* **余弦相似度**衡量两个向量的方向有多接近：方向一致为 1，互相垂直为 0。

![语义搜索的原理](assets/11-semantic-search-principle.svg)

在 store 里，语义搜索分三步：

1. **创建 store 时传入 `index`**：告诉 store 用哪个嵌入模型、对 value 的哪些部分生成向量。
2. **put 时生成向量**：按 `fields` 从 value 中取出文本，用 `embed` 算成向量，存进 `store._vectors`。
3. **search 时比较向量**：用同一个 `embed` 把 query 算成向量，和存好的向量逐个算余弦相似度，按分数从高到低返回。

下面按这三步来看。

**第一步：创建 store 时传入 index**

`InMemoryStore()` 默认不带 index，这时 put 不会生成向量，search 的 `query` 也会被直接忽略。传入 `index` 后，`__init__` 会做两件准备工作：

```python
def __init__(self, *, index: IndexConfig | None = None) -> None:
    ...
    self.index_config = index
    if self.index_config:
        self.index_config = self.index_config.copy()
        # 1. 把 embed 统一包装成 LangChain 的 Embeddings 对象
        self.embeddings = ensure_embeddings(self.index_config.get("embed"))
        # 2. 把 fields 里的每条路径预先拆好，put 时直接用
        self.index_config["__tokenized_fields"] = [
            (p, tokenize_path(p)) if p != "$" else (p, p)
            for p in (self.index_config.get("fields") or ["$"])
        ]
    else:
        self.index_config = None
        self.embeddings = None
```

`index` 的类型是 `IndexConfig`。它是一个 `TypedDict`，即规定了有哪些 key、每个 key 是什么类型的字典，共三个 key：

| key | 是否必填 | 值的要求 | 作用 |
|---|---|---|---|
| `embed` | **必填**，缺了报 `ValueError: embed must be provided` | 四种写法任选一种，见下 | 把文本变成向量 |
| `fields` | 可选，默认 `["$"]` | `list[str]`，每个元素是一条路径（语法见第二步） | 决定对 value 的哪些部分生成向量 |
| `dims` | 可选 | `int`，向量的长度 | InMemoryStore 不使用，见下面的说明 |

`embed` 的四种写法：

```python
"embed": init_embeddings(...)              # ① LangChain 的 Embeddings 对象（举例2）
"embed": embed                             # ② 普通函数（举例1）
"embed": aembed                            # ③ async 函数：只能配合 aput / asearch，用同步的 put 会报 ValueError
"embed": "openai:text-embedding-3-small"   # ④ "提供商:模型名" 字符串：内部调用 init_embeddings(这个字符串)，key 从环境变量读取
```

写成函数时，源码规定的类型是 `Callable[[Sequence[str]], list[list[float]]]`：输入一组文本，输出每段文本对应的向量。`ensure_embeddings` 会把函数包装成 `EmbeddingsLambda` 对象，提供两个方法：

* `embed_documents(texts)`：直接调用你的函数，put 时用；
* `embed_query(text)`：调用 `你的函数([text])[0]`，search 时用。所以即使只算一个 query，函数收到的也是列表。

Python 运行时不检查这个类型，写错了要到 put / search 时才报错。函数必须满足：

| 要求 | 违反时 |
|---|---|
| 返回的向量个数 = 输入的文本个数，按顺序一一对应 | put 时报 `Number of embeddings (…) does not match number of indices (…)` |
| 所有向量的长度相同，包括 query 的向量 | search 时 numpy 报形状不匹配 |

**关于 dims**：`dims` 不会传给嵌入模型，所以不约束模型输出几维；当前版本的 InMemoryStore 和 SqliteStore 也都不读取、不检查它，向量多长都照存。只有 PostgresStore 用它：pgvector 要在建表时声明列类型 `vector(dims)`，之后写入的向量必须正好是 dims 维。举例2 里 `dims` 写的是 512，模型实际输出 1024 维，在 InMemoryStore 里不会报错，换成 PostgresStore 写入时就会被拒绝。

同一份 `index_config`，三种 store 读取的 key 并不相同：

| key | InMemoryStore | SqliteStore | PostgresStore |
|---|---|---|---|
| `embed` | 读 | 读 | 读 |
| `fields` | 读 | **不读**（它读的是 `text_fields`，写 `fields` 会被忽略） | 读 |
| `dims` | 不读 | 不读 | 读，必填 |

举例1：自定义嵌入函数

自定义嵌入函数，目的是查看嵌入向量

In [5]:
from langgraph.store.memory import InMemoryStore

# 自定义嵌入函数
def embed(text: list[str]) -> list[list[float]]:
    return [[1.0] * 6 for _ in range(len(text))]

index_config = {
    "embed": embed, # 使用嵌入函数或嵌入模型赋值
    "dims": 6,
    "fields": ["$", "course"]
}
store = InMemoryStore(
    index = index_config
)

In [9]:
namespace1 = ("users", "Alice", "memories")
key1 = 'preferences'
value1 = {
    "course": "计算机组成原理",
    "sports": "跑步",
    "food": "紫光园奶皮子酸奶"
}

namespace2 = ("users", "Bob", "memories")
key2 = 'preferences'
value2 = {
    "course": "数字电路与模拟电路",
    "sports": "跑步",
    "food": "奶皮子糖葫芦"
}

namespace3 = ("users", "Black", "memories")
key3 = 'preferences'
value3 = {
    "course": "数字电路与模拟电路",
    "sports": "羽毛球",
    "food": "紫光园奶皮子酸奶"
}

store.put(namespace1, key1, value1)
store.put(namespace2, key2, value2)
store.put(namespace3, key3, value3)

from pprint import pprint
pprint(store._vectors)

defaultdict(<function InMemoryStore.__init__.<locals>.<lambda> at 0x11276b9c0>,
            {('users', 'Alice', 'memories'): defaultdict(<class 'dict'>,
                                                         {'preferences': {'$': [1.0,
                                                                                1.0,
                                                                                1.0,
                                                                                1.0,
                                                                                1.0,
                                                                                1.0],
                                                                          'course': [1.0,
                                                                                     1.0,
                                                                                     1.0,
                                                                  

In [11]:
from pprint import pprint
pprint(store._vectors[('users', 'Alice', 'memories')]['preferences']['$'])
from pprint import pprint
pprint(store._vectors[('users', 'Alice', 'memories')]['preferences']['course'])

[1.0, 1.0, 1.0, 1.0, 1.0, 1.0]
[1.0, 1.0, 1.0, 1.0, 1.0, 1.0]


**第二步：put 时，三个 key 如何把 value 变成向量**

![put 时 index_config 三个 key 的分工](assets/12-index-config-put-flow.svg)

put 交给 `batch()` 后，InMemoryStore 依次执行：

```python
to_embed = self._extract_texts(put_ops)                               # ① 用 fields 从 value 取文本
if to_embed and self.index_config and self.embeddings:
    embeddings = self.embeddings.embed_documents(list(to_embed))      # ② 用 embed 把文本变成向量
    self._insertinmem_store(to_embed, embeddings)                     # ③ 按路径存进 _vectors
self._apply_put_ops(put_ops)                                          # 最后把 Item 存进 _data（3.2.3 讲过）
```

三个 key 的作用：

* **fields：取文本。** 对 fields 里的每条路径调用 `get_text_at_path(value, 路径)`，取出该位置的文本。值不是字符串时会先转换：数字变成 `"20"` 这样的字符串，字典和列表变成 JSON 字符串。`"$"` 表示整个 value，用 `json.dumps(value, sort_keys=True, ensure_ascii=False)` 转成一个字符串，键按字母排序。
* **embed：算向量。** 一次 put 取出的所有文本放进一个列表，**只调用一次** `embed_documents`，返回同样个数的向量。
* **dims：不参与。** embed 返回几维就存几维。

所以举例1 中 `fields=["$", "course"]` 时，put Alice 这条记忆，embed 收到的是：

```python
['{"course": "计算机组成原理", "food": "紫光园奶皮子酸奶", "sports": "跑步"}', '计算机组成原理']
```

返回的 2 个向量以路径为键存进 `_vectors`，这就是上面打印出 `'$'` 和 `'course'` 两个向量的原因：

```python
_vectors[("users", "Alice", "memories")]["preferences"] = {"$": [...], "course": [...]}
#          namespace                         key            路径   向量
```

**fields 的路径语法**

fields 的每个元素是一条“路径”，告诉 store 去 value 的哪个位置取文本。以下面这个 value 为例：

```python
value = {
    "course": "计算机组成原理",
    "tags": ["跑步", "羽毛球"],              # 值是列表
    "profile": {"city": "北京", "age": 20},  # 值是字典
}
```

| fields 里写 | 取的是 | 取出的文本 | 向量个数 |
|---|---|---|---|
| `"$"` | 整个 value | `'{"course": "计算机组成原理", "profile": {…}, "tags": […]}'` | 1 |
| `"course"` | 顶层键 `course` 的值 | `"计算机组成原理"` | 1 |
| `"tags"` | 整个列表，转成 JSON 字符串 | `'["跑步", "羽毛球"]'` | 1 |
| `"tags[*]"` | 列表里的**每个**元素 | `"跑步"`、`"羽毛球"` | **2** |
| `"tags[0]"` | 列表的第 1 个元素（下标从 0 开始） | `"跑步"` | 1 |
| `"profile"` | 整个字典，转成 JSON 字符串 | `'{"age": 20, "city": "北京"}'` | 1 |
| `"profile.city"` | 用 `.` 进入下一层，取 `profile` 里的 `city` | `"北京"` | 1 |
| `"profile.age"` | 数字转成字符串 | `"20"` | 1 |
| `"hobby"` | 键不存在 | 无 | 0 |

`"tags[*]"` 取出几段文本就生成几个向量，存在 `tags[*].0`、`tags[*].1` 这样的路径下（末尾是元素下标），所以每条记忆的向量个数可以不同。

> 注意：InMemoryStore 会把同一次 put 里内容相同的文本去重后再交给 embed。如果两段文本完全相同，例如 `{"tags": ["跑步", "跑步"]}` 配 `fields=["tags[*]"]`，embed 只返回 1 个向量，却要填 2 个位置，put 会报 `ValueError: Number of embeddings (1) does not match number of indices (2)`。SqliteStore / PostgresStore 不去重，没有这个问题。

下面用一个会打印输入的 embed 验证：

In [ ]:
# 会打印自己收到什么的 embed，返回的 2 维向量没有实际含义，只为演示
def spy_embed(texts: list[str]) -> list[list[float]]:
    print("embed 收到:", texts)
    return [[float(len(t)), 1.0] for t in texts]

demo = InMemoryStore(index={
    "embed": spy_embed,
    "fields": ["tags[*]", "profile.city", "profile.age", "hobby"],
})
demo.put(("users", "demo"), "m1", {
    "course": "计算机组成原理",
    "tags": ["跑步", "羽毛球"],
    "profile": {"city": "北京", "age": 20},
})
print("存下的向量:", dict(demo._vectors[("users", "demo")]["m1"]))

embed 收到: ['跑步', '羽毛球', '北京', '20']
存下的向量: {'tags[*].0': [2.0, 1.0], 'tags[*].1': [3.0, 1.0], 'profile.city': [2.0, 1.0], 'profile.age': [2.0, 1.0]}


举例2: 嵌入模型初始化

In [14]:
import os
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI
from langchain.agents import create_agent
from langchain.agents.structured_output import ToolStrategy
from langchain.tools import tool
from langchain.embeddings import init_embeddings

load_dotenv(override=True, verbose=True, dotenv_path="/Users/skk/Developer/lang-chain-learn/conf/.env")



embedding_model = init_embeddings(
    model="openai:Qwen3-VL-Embedding-2B",
    api_key=os.environ["embedding_model_key"],
    base_url=os.environ["embedding_model_base"],
    #profile=custom_profile
)

index_config = {
    "embed": embedding_model,
    "dims": 512,
    "fields": ["$"]
}

store = InMemoryStore(
    index = index_config
)

In [16]:
namespace1 = ("users", "Alice", "memories")
key1 = 'preferences'
value1 = {
    "course": "计算机组成原理",
    "sports": "跑步",
    "food": "紫光园奶皮子酸奶"
}

namespace2 = ("users", "Bob", "memories")
key2 = 'preferences'
value2 = {
    "course": "数字电路与模拟电路",
    "sports": "跑步",
    "food": "奶皮子糖葫芦"
}

namespace3 = ("users", "Black", "memories")
key3 = 'preferences'
value3 = {
    "course": "数字电路与模拟电路",
    "sports": "羽毛球",
    "food": "紫光园奶皮子酸奶"
}

store.put(namespace1, key1, value1)
store.put(namespace2, key2, value2)
store.put(namespace3, key3, value3)

In [ ]:
from pprint import pprint
pprint(store._vectors[('users', 'Alice', 'memories')]['preferences']['$'])

[0.0133056640625,
 -0.0106048583984375,
 -0.0229339599609375,
 -0.015655517578125,
 0.0183868408203125,
 0.0171966552734375,
 -0.036712646484375,
 -0.0143890380859375,
 0.01183319091796875,
 -0.003444671630859375,
 0.031219482421875,
 -0.00447845458984375,
 0.0134735107421875,
 -0.0159912109375,
 0.02947998046875,
 -0.003459930419921875,
 -0.0244903564453125,
 0.048187255859375,
 0.07708740234375,
 0.01451873779296875,
 -0.025970458984375,
 0.040283203125,
 0.02880859375,
 -0.0145416259765625,
 -0.01557159423828125,
 -0.0156402587890625,
 -0.042022705078125,
 0.006725311279296875,
 0.020172119140625,
 -0.0478515625,
 0.033905029296875,
 -0.0242462158203125,
 0.017578125,
 -0.0116119384765625,
 0.02947998046875,
 -0.007762908935546875,
 0.017608642578125,
 0.0193328857421875,
 0.01322174072265625,
 0.010284423828125,
 -0.0038928985595703125,
 0.0207061767578125,
 0.0162353515625,
 -0.024627685546875,
 0.009033203125,
 0.0281219482421875,
 0.01445770263671875,
 -0.09039306640625,
 0.0015

上面打印出的向量共 **1024** 个数，而 `index_config` 里 `dims` 写的是 512：`dims` 不约束模型的输出维度，InMemoryStore 也不检查（见第一步的说明）。向量的维度只由模型决定，和文本长短无关，整段 JSON 和一个词得到的向量一样长。

补充：`init_embeddings("openai:...")` 返回的是 `OpenAIEmbeddings`。它默认不发送原文，而是先用 OpenAI 的 tiktoken 把文本转成 token id 再发给 `base_url`。很多非 OpenAI 的服务不认这种输入（LangChain 文档建议这时传 `check_embedding_ctx_length=False` 改发原文）。当前用的中转服务能正确处理 token id：对比过两种方式，得到的向量余弦相似度为 1.0000。

**第三步：search 时如何用向量检索**

search 带 query 时，InMemoryStore 的处理顺序（源码 `_filter_items`、`_embed_search_queries`、`_batch_search`）：

1. **选候选**：先按 `namespace_prefix` 和 `filter` 选出候选记忆，连同它们在 `_vectors` 里的向量一起取出。
2. **query 变向量**：调用 `embed_query(query)`，即 `embed([query])[0]`。它和 put 用的是同一个 embed，维度一致才能比较。
3. **算分**：query 向量和每个候选的每个向量算余弦相似度，公式是 `cos = (a·b) / (|a| × |b|)`：两个向量逐位相乘再求和，除以两个向量长度的乘积。
4. **每条记忆取最高分**：一条记忆有多个向量时（fields 有多条路径，或用了 `[*]`），只保留最高的那个分数，记忆本身只返回一次。
5. **排序、分页**：按分数从高到低排列，再按 `offset`、`limit` 截取，分数写进 `SearchItem.score`。

第 4、5 步的源码：

```python
scores = _cosine_similarity(query_embedding, flat_vectors)   # query 和所有候选向量的相似度
sorted_results = sorted(zip(scores, flat_items), key=lambda x: x[0], reverse=True)
seen = set()
kept = []
for score, item in sorted_results:          # 已经按分数从高到低
    key = (item.namespace, item.key)
    if key in seen:
        continue                            # 同一条记忆后面的分数更低，跳过
    ix = len(seen)
    seen.add(key)
    if ix >= op.offset + op.limit:
        break
    if ix < op.offset:
        continue
    kept.append((score, item))
```

search **不会按分数筛掉不相关的记忆**，只负责排序：候选有几条就返回几条（最多 `limit` 条），分数再低也会返回。

In [19]:
for item in store.search(("users", ), query="数电模电"):
    print(item)

Item(namespace=['users', 'Bob', 'memories'], key='preferences', value={'course': '数字电路与模拟电路', 'sports': '跑步', 'food': '奶皮子糖葫芦'}, created_at='2026-10-08T14:07:38.178384+00:00', updated_at='2026-10-08T14:07:38.178389+00:00', score=0.5771769785025985)
Item(namespace=['users', 'Black', 'memories'], key='preferences', value={'course': '数字电路与模拟电路', 'sports': '羽毛球', 'food': '紫光园奶皮子酸奶'}, created_at='2026-10-08T14:07:38.479198+00:00', updated_at='2026-10-08T14:07:38.479207+00:00', score=0.563490223105138)
Item(namespace=['users', 'Alice', 'memories'], key='preferences', value={'course': '计算机组成原理', 'sports': '跑步', 'food': '紫光园奶皮子酸奶'}, created_at='2026-10-08T14:07:37.868887+00:00', updated_at='2026-10-08T14:07:37.868894+00:00', score=0.4953971450653457)


上面的结果：举例2 用的是 `fields=["$"]`，每条记忆只有 1 个向量，来自整段 JSON。

* query “数电模电”是“数字电路与模拟电路”的简称，字面上并不相同，但 Bob 和 Black 的 course 正是这门课，分数最高（0.577、0.563）；Alice 学的是计算机组成原理，分数最低（0.495）。
* Bob 和 Black 的 course 相同，分数却不同：向量来自整段 JSON，sports、food 也参与了计算。
* 真实模型的分数都挤在 0.5 左右，只有相对高低有意义，不能把 0.5 理解为“一半相似”。

下面用手写的 2 维向量验证“每条记忆取最高分”：Alice 有两个向量，和 query “游泳”的相似度分别是 0.994（跑步）和 0.685（羽毛球），最后只保留 0.994。

In [ ]:
# 手写的 2 维向量：第 1 维表示“运动”，第 2 维表示“食物”
VECTORS = {
    "跑步": [1.0, 0.0],
    "羽毛球": [0.6, 0.8],
    "糖葫芦": [0.0, 1.0],
    "游泳": [0.9, 0.1],
}

def toy_embed(texts: list[str]) -> list[list[float]]:
    return [VECTORS[t] for t in texts]

demo = InMemoryStore(index={"embed": toy_embed, "fields": ["tags[*]"]})
demo.put(("users", "Alice"), "m1", {"tags": ["跑步", "羽毛球"]})
demo.put(("users", "Bob"), "m1", {"tags": ["糖葫芦"]})

for item in demo.search(("users",), query="游泳"):
    print(item.namespace, item.value, "score =", round(item.score, 3))

('users', 'Alice') {'tags': ['跑步', '羽毛球']} score = 0.994
('users', 'Bob') {'tags': ['糖葫芦']} score = 0.11
